# Feed-Forward Neural Networks

Let's now consider another class of models within the ERM framework: **feed-forward neural networks**.

As a point of reference, consider two of our basic models:

1. **Linear regression**, where the goal was to predict a continuous outcome

   $$
   y \in \mathbb{R}
   $$

   from an input

   $$
   x \in \mathbb{R}^D.
   $$

   We chose the class of linear score functions

   $$
   \mathcal{S} = \{s_w(x) = w^\top x : w \in \mathbb{R}^D\},
   $$

   and, using squared error loss, arrived at the least squares problem.

2. **Logistic regression**, where the goal was to predict a categorical outcome

   $$
   y \in \{1,\ldots,K\}.
   $$

   We again used a linear score function

   $$
   s_W(x) = W^\top x,
   $$

   but then mapped the score to probabilities

    $$
    p_W(x) = \mathrm{softmax}(W^\top x).
   $$


   and optimized $W$ using the categorical cross-entropy loss.


Neural networks fit into this same setup. The main difference is that we will replace the simple linear score function with a more flexible function $s_\theta$. At a high level, the neural network score function is built by composing several simpler functions. For example, we might write

$$
s_\theta(x)=
s_L(s_{L-1}(\cdots s_2(s_1(x)) \cdots)).
$$

Here each function $s_i$ represents one **layer** of the network. Each layer has its own parameters. The full parameter vector for the network is then the collection of all layer-specific parameters:

$$
\theta = (\theta_1, \theta_2, \dots, \theta_L).
$$

This composition structure is the key difference from linear and logistic regression. In linear and logistic regression, the score function was a single linear function of the input. In a neural network, the score function is built by passing the input through a sequence of learned transformations.

In the simplest feed-forward networks, each layer applies an affine transformation followed by a nonlinear activation function. We will make this precise shortly. For now, the important point is that $s_\theta(x)$ is not just a single weighted sum of the original features. It is a layered function whose intermediate representations are learned from the data.

So the ERM setup becomes

$$
\hat{\theta} = \arg\min_\theta \hat{R}(\theta),
$$

where

$$
\hat{R}(\theta) = \frac{1}{N}\sum_{n=1}^N \ell(y_n, s_\theta(x_n)).
$$

For some loss function $\ell$. Typically we'll use squared-error for regression and categorical cross entropy for classification tasks.

## Feature Learning

Linear and logistic regression are often quite useful. They are simple, interpretable, and relatively easy to fit. But they also have a major limitation: the score function is linear in the input representation. There are two broad ways to address this.

The first approach is **feature engineering**. We can manually construct new features and put them into the design matrix $X$. For example, if the raw input has two components,

$$
x = (x_1, x_2),
$$

we might construct additional features like

$$
x_1^2, \quad x_2^2, \quad x_1x_2.
$$

Then a linear model in this expanded feature representation can represent nonlinear relationships in the original raw inputs.

The second approach is to let the model **learn the representation itself**. This is the basic idea behind neural networks.

Instead of manually specifying all of the transformed features, we define a model that learns intermediate features from the data. These learned features are then combined to make predictions. In a neural network, the model learns new features internally and then uses those learned features for prediction.

Recall that we wrote the neural network score as

$$
s_\theta(x)=
s_L(s_{L-1}(\cdots s_2(s_1(x)) \cdots)).
$$

We can think of each intermediate layer as producing a new representation of the input. The first layer takes the original features $x$ and transforms them into a new set of (so-called *hidden*) features,

$$
h^{(1)} = s_1(x).
$$

The second layer takes this learned representation and transforms it again,

$$
h^{(2)} = s_2(h^{(1)}).
$$

Continuing this way,

$$
h^{(i)} = s_i(h^{(i-1)}).
$$

The final layer then uses the last learned representation to produce the score,

$$
s_\theta(x) = s_L(h^{(L-1)}).
$$

So the network is not just applying one model directly to the original inputs. It is constructing a sequence of representations,

$$
x \mapsto h^{(1)} \mapsto h^{(2)} \mapsto \cdots \mapsto h^{(L-1)} \mapsto s_\theta(x).
$$

This is the sense in which the network learns features internally. Each hidden layer produces features that are learned from the data, and the final layer combines those learned features to make a prediction.

## What Is the Form of Each Layer?

The next question is: what is the form of each layer $s_i$?

In a standard feed-forward neural network, each hidden layer usually has two parts:

1. an affine transformation, followed by
2. a nonlinear **activation function**

That is, a typical hidden layer has the form

$$
s_i(h) = \phi_i(W^{(i)}h + b^{(i)}).
$$

Here:

- $h$ is the input to the layer
- $W^{(i)}$ is a matrix of weights
- $b^{(i)}$ is a vector of bias/intercept terms
- $\phi_i$ is the activation function used at layer $i$

The subscript on $\phi_i$ allows different layers to use different activation functions, although in many basic feed-forward networks we use the same activation function for every hidden layer.

So each layer first forms weighted combinations of the previous layer's features, then applies a nonlinear transformation.

Suppose the input to layer $i$ has dimension $d_{i-1}$, and the output of layer $i$ has dimension $d_i$. That is,

$$
h \in \mathbb{R}^{d_{i-1}}
$$

and

$$
s_i(h) \in \mathbb{R}^{d_i}.
$$

Then the weight matrix and bias vector have sizes

$$
W^{(i)} \in \mathbb{R}^{d_i \times d_{i-1}},
$$

and

$$
b^{(i)} \in \mathbb{R}^{d_i}.
$$

Therefore,

$$
W^{(i)}h + b^{(i)} \in \mathbb{R}^{d_i}.
$$

The activation function $\phi_i$ is usually applied componentwise, so it maps

$$
\phi_i : \mathbb{R}^{d_i} \to \mathbb{R}^{d_i}.
$$

Thus,

$$
s_i(h)=
\phi_i(W^{(i)}h + b^{(i)})
\in \mathbb{R}^{d_i}.
$$

For the first layer, the input is the original feature vector $x$:

$$
h^{(1)}=
\phi_1(W^{(1)}x + b^{(1)}).
$$

For the next layer, the input is the learned representation from the previous layer:

$$
h^{(2)}=
\phi_2(W^{(2)}h^{(1)} + b^{(2)}).
$$

Continuing this way,

$$
h^{(i)}=
\phi_i(W^{(i)}h^{(i-1)} + b^{(i)}).
$$

So the network builds a sequence of learned representations:

$$
x \mapsto h^{(1)} \mapsto h^{(2)} \mapsto \cdots.
$$

The layer-specific parameter vector $\theta_i$ consists of all entries of the weight matrix $W^{(i)}$ and the bias vector $b^{(i)}$.

Typically, the activation function $\phi_i$ does not have parameters associated with it. In that case,

$$
\theta_i = (W^{(i)}, b^{(i)}).
$$

The full network parameter vector $\theta$ is the collection of the parameters from all layers:

$$
\theta = (\theta_1, \theta_2, \dots, \theta_L).
$$

Common choices for the activation function include:

- **ReLU**:

$$
\phi_i(t) = \max\{0,t\}
$$

- **Sigmoid**:

$$
\phi_i(t) = \frac{1}{1 + e^{-t}}
$$

- **Hyperbolic tangent**:

$$
\phi_i(t) = \tanh(t)
$$

- **Identity activation**:

$$
\phi_i(t) = t
$$

The identity activation is often used in the output layer for regression. Nonlinear activations such as ReLU, sigmoid, or tanh are used in hidden layers to make the network more flexible than a purely linear model. If we don't include any activation functions then the network will essentially reduce to a very complicated representation of a linear score. 

## Why Are They Called "Networks"?

They are called **neural networks** because the model is organized as a network of connected units.

Each unit, or node, takes inputs from the previous layer, forms a weighted combination of those inputs, applies an activation function, and passes the result forward to the next layer. The connections between units are the weights. The values at the hidden units are intermediate learned features.

For example, in a feed-forward network, information flows through the model as

$$
x \mapsto h^{(1)} \mapsto h^{(2)} \mapsto \cdots \mapsto s_\theta(x).
$$

This looks like a network because each layer contains several units, and each unit is connected to units in the next layer. The network is called **feed-forward** because the information moves in one direction: from the input layer, through the hidden layers, to the output layer.

The word "neural" comes from a loose analogy with biological neurons, but in this course we should think of these as mathematical models: compositions of weighted sums and nonlinear activation functions.

An example:

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Network architecture:
# 3 input features, 4 units in hidden layer 1,
# 3 units in hidden layer 2, 1 output
layer_sizes = [3, 4, 3, 1]
layer_labels = [
    "Input layer\n$x$",
    "Hidden layer 1\n$h^{(1)}$",
    "Hidden layer 2\n$h^{(2)}$",
    "Output\n$s_\\theta(x)$"
]

fig, ax = plt.subplots(figsize=(9, 4.8))

# Horizontal position of each layer
x_positions = np.arange(len(layer_sizes))

# Store node coordinates
node_positions = []

for i, size in enumerate(layer_sizes):
    # Vertically center the nodes in each layer
    y_positions = np.linspace(-(size - 1) / 2, (size - 1) / 2, size)
    layer_nodes = [(x_positions[i], y) for y in y_positions]
    node_positions.append(layer_nodes)

# Draw connections between adjacent layers
for i in range(len(layer_sizes) - 1):
    for x1, y1 in node_positions[i]:
        for x2, y2 in node_positions[i + 1]:
            ax.plot([x1, x2], [y1, y2], linewidth=0.8, alpha=0.45)

# Draw nodes and labels
for i, layer_nodes in enumerate(node_positions):
    xs = [p[0] for p in layer_nodes]
    ys = [p[1] for p in layer_nodes]

    ax.scatter(xs, ys, s=650, zorder=3)

    for j, (x, y) in enumerate(layer_nodes, start=1):
        if i == 0:
            label = f"$x_{j}$"
        elif i == len(layer_sizes) - 1:
            label = "$s_\\theta(x)$"
        else:
            label = f"$h^{{({i})}}_{j}$"

        ax.text(x, y, label, ha="center", va="center", fontsize=10, zorder=4)

# Add layer labels underneath
for i, label in enumerate(layer_labels):
    ax.text(x_positions[i], -2.55, label, ha="center", va="top", fontsize=11)

# Add feed-forward direction arrow
ax.annotate(
    "feed-forward direction",
    xy=(2.75, 2.1),
    xytext=(0.25, 2.1),
    arrowprops=dict(arrowstyle="->", linewidth=1.5),
    ha="center",
    va="center",
    fontsize=11
)

ax.set_xlim(-0.5, len(layer_sizes) - 0.5)
ax.set_ylim(-3.0, 2.7)
ax.axis("off")
fig.tight_layout()

plt.show()

Two common ways to describe the size of a neural network are its **depth** and its **width**.

The **depth** of the network refers to the number of layers in the composition. For example, if

$$
s_\theta(x)=
s_L(s_{L-1}(\cdots s_2(s_1(x)) \cdots)),
$$

then the network has depth $L$, depending on whether we are counting all layers in the composition. Sometimes people count only the hidden layers, so it is useful to be explicit about the convention.

The **width** of layer $i$ refers to the number of units in that layer. In our notation, if

$$
s_i : \mathbb{R}^{d_{i-1}} \to \mathbb{R}^{d_i},
$$

then layer $i$ has width $d_i$.

The quantity $d_{i-1}$ is the width of the previous layer, and $d_i$ is the width of the current layer. Since

$$
W^{(i)} \in \mathbb{R}^{d_i \times d_{i-1}},
$$

the number of weights in layer $i$ is

$$
d_i d_{i-1}.
$$

There are also $d_i$ bias terms, since

$$
b^{(i)} \in \mathbb{R}^{d_i}.
$$

So the total number of parameters in layer $i$ is

$$
d_i d_{i-1} + d_i.
$$

Thus, wider layers usually mean more parameters, and deeper networks mean more layers of learned transformations.

## One-hidden-layer Network

Consider a simple one-hidden-layer network. Suppose the input is

$$
x \in \mathbb{R}^D.
$$

The network has $H$ hidden units. The hidden layer is defined by

$$
h = \phi(Wx + b),
$$

where

$$
W \in \mathbb{R}^{H \times D},
$$

$$
b \in \mathbb{R}^H,
$$

and therefore

$$
h \in \mathbb{R}^H.
$$

The first hidden **unit** computes

$$
h_1 = \phi(w_1^\top x + b_1),
$$

the second hidden unit computes

$$
h_2 = \phi(w_2^\top x + b_2),
$$

and so on, up to

$$
h_H = \phi(w_H^\top x + b_H).
$$

Here $w_j^\top$ is the $j$th row of $W$. Each hidden unit takes a linear combination of the input features and then applies the activation function.

Stacking the hidden units together gives

$$
h =
\begin{pmatrix}
h_1 \\
h_2 \\
\vdots \\
h_H
\end{pmatrix}
\in \mathbb{R}^H.
$$

So the hidden layer transforms the original input vector $x \in \mathbb{R}^D$ into a learned feature vector $h \in \mathbb{R}^H$.

The output layer then depends on the prediction task.

### Regression Output

For regression, we usually want a single real-valued prediction. So the output layer (the last layer) maps

$$
\mathbb{R}^H \to \mathbb{R}.
$$

A common choice is a linear output layer:

$$
s_\theta(x) = v^\top h + c,
$$

where

$$
v \in \mathbb{R}^H
$$

and

$$
c \in \mathbb{R}.
$$

Thus,

$$
s_\theta(x) \in \mathbb{R}.
$$

This is equivalent to choosing a *linear* activation for this last layer. 

All together, then:

$$
s_\theta(x) = v^\top \phi(Wx + b) + c.
$$

So for regression, a one-hidden-layer network first transforms $x$ into learned features,

$$
h = \phi(Wx + b),
$$

and then applies a linear regression-style output layer to those learned features.

### Classification Output

For classification with $K$ classes, we want the network to output one predicted probability for each class.

To do this, we choose the final layer to have width $K$. If the hidden representation is

$$
h \in \mathbb{R}^H,
$$

then the final affine transformation has the form

$$
Vh + c,
$$

where

$$
V \in \mathbb{R}^{K \times H}
$$

and

$$
c \in \mathbb{R}^K.
$$

Thus,

$$
Vh+c \in \mathbb{R}^K.
$$

These $K$ values are class scores, also called logits. To convert them into predicted class probabilities, we use the **softmax activation function**, so that the final layer all together is:

$$
s_\theta(x)=\text{softmax}(Vh+c).
$$

and all together we have:

$$
s_\theta(x)=\text{softmax}(V\phi(Wx+b)+c).
$$


The final output $s_\theta(x)$ is a vector in $\mathbb{R}^K$, where each entry is nonnegative and the entries sum to one. (i.e. they're probabilities).

One small notational point is worth mentioning. Earlier, in logistic regression, we used the word **score** for the value before applying the sigmoid function. Here, for classification networks, we are instead writing $s_\theta(x)$ for the final output after applying softmax.

This is just a convention. The intermediate values

$$
Vh + c
$$

are often called **logits** or **class scores**, while

$$
s_\theta(x) = \text{softmax}(Vh+c)
$$

is the vector of predicted probabilities. Later, when we define the loss function for classification, we will make sure it matches this convention.

The main point is that the final output layer usually depends on the task:

- For regression, the standard choice is a **linear output layer**.
- For classification, the standard choice is a **softmax output layer**.

These are the usual default choices because regression requires real-valued predictions, while classification requires predicted class probabilities.

## Loss Functions and ERM

Once we have defined the network architecture defining $s_\theta(x)$, we still need to choose a loss function. The loss function should match the prediction task.

### Regression

For regression, the target is continuous:

$$
y \in \mathbb{R}.
$$

The network output is also a real number:

$$
s_\theta(x) \in \mathbb{R}.
$$

A common choice is squared loss:

$$
\ell(y, s_\theta(x))=
(y - s_\theta(x))^2.
$$

Then the empirical risk is

$$
\hat{R}(\theta)=
\frac{1}{N}
\sum_{n=1}^N
(y_n - s_\theta(x_n))^2.
$$

### Classification

For classification with $K$ classes, the network output is a vector of predicted class probabilities:

$$
s_\theta(x)=p_\theta(x)
\in \mathbb{R}^K.
$$

Since we are using the convention that $s_\theta(x)$ is the output **after** the softmax activation, the entries satisfy

$$
s_{\theta,k}(x) \ge 0
$$

and

$$
\sum_{k=1}^K s_{\theta,k}(x) = 1.
$$

Like multi-class logistic regression, a common loss function for multiclass classification is **categorical cross-entropy**.

If the true class label is $y \in \{1,\dots,K\}$, then the loss is

$$
\ell(y, s_\theta(x))=
-\log(s_{\theta,y}(x)).
$$

Equivalently, if the outcome is represented as a one-hot vector

$$
t =
\begin{pmatrix}
t_1 \\
t_2 \\
\vdots \\
t_K
\end{pmatrix},
$$

where $t_k = 1$ for the correct class and $t_k = 0$ otherwise, then categorical cross-entropy can be written as

$$
\ell(y, s_\theta(x))=
-\sum_{k=1}^K t_k \log(s_{\theta,k}(x)).
$$

### ERM Problem

In either case, the overall framework is unchanged. We choose parameters by minimizing empirical risk:

$$
\hat{\theta}=
\arg\min_\theta
\hat{R}(\theta),
$$

where

$$
\hat{R}(\theta)=
\frac{1}{N}
\sum_{n=1}^N
\ell(y_n, s_\theta(x_n)).
$$

The model class has changed from a linear score function to a neural network, but the learning problem is still an ERM problem.